# Reproduce the rolling LightGBM submission

This notebook runs the repository's Python modules for the historical LightGBM
experiment. The original research record is
[SKS_xgboost_moving_avg_minimal.ipynb](SKS_xgboost_moving_avg_minimal.ipynb).
The archived submission received **1.11764** on Kaggle's private leaderboard.
The comparison below measures agreement with that CSV.

The full fit uses about 23.8 million training rows and requires substantial
memory and time. Run from the first cell using the Python kernel from the
locked `uv` environment. Choose a fresh output path before rerunning.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
root = next(
    (path for path in (cwd, *cwd.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Start Jupyter from the repository or its notebooks directory")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_loading import (
    load_rolling_metadata,
    load_rolling_test_inputs,
    load_training_data,
)
from src.evaluation import compare_submission_reference, validate_reference_ids
from src.features import add_rolling_sales_features, build_rolling_test_history
from src.prediction import predict_rolling_submission, write_submission
from src.preprocessing import (
    clean_rolling_training_sales,
    prepare_rolling_test_data,
    prepare_rolling_training_data,
)
from src.training import fit_rolling_lightgbm

## Load and clean the data

Keep sales from January 2017 onward, clip returns at zero, and apply the
original log transform, type casts, and sort order. The archived CSV is optional.

In [ ]:
filename = "lightgbm_mean_lag3_7_14_21_with_features.csv"
data_dir = root / "data" / "raw" / "favorita-grocery-sales-forecasting"
output_path = root / "data" / "reproduction" / filename
reference_path = root / "data" / "submission" / filename

In [ ]:
if output_path.exists():
    raise FileExistsError(f"Choose a new output path. {output_path} already exists")

test, sample = load_rolling_test_inputs(data_dir)
if reference_path is not None and reference_path.is_file():
    validate_reference_ids(reference_path, sample["id"])

stores, items, oil = load_rolling_metadata(data_dir)
training = clean_rolling_training_sales(load_training_data(data_dir / "train.csv"))
print(f"Loaded {len(training):,} training rows and {len(test):,} test rows.")

## Build the historical features

Compute observation lags of 3, 7, and 14 and rolling means over 3, 7, 14, and 21
observations for each store and item. Test features use the last 60 days of
training history and unlabeled test rows. Join store, item, and date-specific
oil data and derive the calendar fields.

The original training rolling means include the current target, and the test
join includes oil values dated after the forecast origin. These are historical
reproduction choices documented in the [baseline audit](../docs/BASELINE.md).

In [ ]:
add_rolling_sales_features(training)
test_history = build_rolling_test_history(training, test)
del test

features, target, weights = prepare_rolling_training_data(training, stores, items, oil)
del training

## Fit LightGBM

Use the original preprocessing and LightGBM parameters. The model fits all
retained rows with perishable-item weights. The original experiment used no
local validation split.

In [ ]:
pipeline = fit_rolling_lightgbm(features, target, weights)
del features, target, weights

## Predict, compare, and export

Clip predictions on the log scale, apply the inverse transform, and keep the
original positional assignment to the sample submission IDs. The comparison
reports exact and tolerance-based agreement when the archived CSV is present.
Kaggle's private labels are unavailable locally.

In [ ]:
test_features = prepare_rolling_test_data(test_history, stores, items, oil)
del test_history
submission = predict_rolling_submission(pipeline, test_features, sample)
del test_features

write_submission(submission, sample["id"], output_path)
print(f"Wrote {len(submission):,} forecasts to {output_path}")
if reference_path is not None and reference_path.is_file():
    print(compare_submission_reference(output_path, reference_path))
submission.head()